# Day 016 — Project Solution: Streaming Chatbot

A scripted demo of a streaming multi-turn chatbot with a custom persona.
No `input()` — questions are hard-coded so the gate can execute this notebook.

In [ ]:
import ollama
import io
import sys

In [ ]:
# ── All five functions from today's lessons ──────────────────────────

def stream_tokens(messages: list[dict], model: str = "llama3.2"):
    """Yield tokens one at a time from a streaming Ollama response."""
    response = ollama.chat(model=model, messages=messages, stream=True)
    for chunk in response:
        yield chunk["message"]["content"]


def collect_stream(tokens) -> str:
    """Consume a token iterator and return the full reply as a string."""
    return "".join(tokens)


def print_stream(tokens) -> None:
    """Print tokens to stdout as they arrive; final newline at end."""
    for token in tokens:
        print(token, end="", flush=True)
    print()


def append_turn(history: list[dict], user_text: str, assistant_text: str) -> list[dict]:
    """Return a new history list with one user+assistant turn appended."""
    return history + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]


def stream_chat_turn(
    history: list[dict],
    user_input: str,
    model: str = "llama3.2",
) -> tuple[str, list[dict]]:
    """Stream the model reply to stdout and return (reply, new_history)."""
    messages = history + [{"role": "user", "content": user_input}]
    tokens = stream_tokens(messages, model)
    parts = []
    for token in tokens:
        print(token, end="", flush=True)
        parts.append(token)
    print()
    reply = "".join(parts)
    return reply, append_turn(history, user_input, reply)


def truncate_history(history: list[dict], max_turns: int = 10) -> list[dict]:
    """Keep the system prompt and the last max_turns*2 non-system messages."""
    if not history:
        return []
    if history[0]["role"] == "system":
        system, tail = [history[0]], history[1:]
    else:
        system, tail = [], history
    return system + tail[-(max_turns * 2):]


def reset_history(history: list[dict]) -> list[dict]:
    """Return a new history containing only the system prompt (if present)."""
    if history and history[0]["role"] == "system":
        return [history[0]]
    return []


def format_history(history: list[dict]) -> str:
    """Render conversation history as a readable transcript."""
    labels = {"user": "You", "assistant": "Bot"}
    lines = []
    for msg in history:
        if msg["role"] == "system":
            continue
        label = labels.get(msg["role"], msg["role"].capitalize())
        lines.append(f"{label}: {msg['content']}")
    return "\n".join(lines)

In [ ]:
# ── Custom persona ───────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are Sage, a calm and curious AI tutor. You explain concepts clearly "
    "with concrete examples and always check whether the student understood "
    "before moving on. Keep answers brief — three sentences maximum unless "
    "the student asks for more detail."
)


def run_streaming_chatbot(
    system_prompt: str = SYSTEM_PROMPT,
    model: str = "llama3.2",
    max_turns: int = 10,
) -> None:
    """Run a streaming multi-turn CLI chatbot until the user types /quit."""
    history = [{"role": "system", "content": system_prompt}]
    print("Streaming chatbot ready. Commands: /quit  /reset  /history")
    print("-" * 50)

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user_input:
            continue

        if user_input.startswith("/"):
            if user_input == "/quit":
                print("Goodbye!")
                break
            elif user_input == "/reset":
                history = reset_history(history)
                print("Bot: Conversation reset.")
            elif user_input == "/history":
                transcript = format_history(history)
                print(transcript if transcript else "(no history yet)")
            else:
                print(f"Bot: Unknown command: {user_input}")
            continue

        print("Bot: ", end="", flush=True)
        reply, history = stream_chat_turn(history, user_input, model)
        history = truncate_history(history, max_turns)

## Scripted Demo

The gate executes this notebook — `input()` is not used here. The demo runs three hard-coded turns to verify streaming works end-to-end.

In [ ]:
# Scripted streaming demo — no input(), safe for gate execution
questions = [
    "What is the capital of France?",
    "Explain in one sentence why the sky is blue.",
    "What is 12 times 8?",
]

history = [{"role": "system", "content": SYSTEM_PROMPT}]

for question in questions:
    print(f"You: {question}")
    print("Bot: ", end="", flush=True)
    reply, history = stream_chat_turn(history, question)
    history = truncate_history(history, max_turns=10)
    print()  # blank line between turns

print("--- Conversation history ---")
print(format_history(history))